# Notebook 12: Metric Sample-Size ($M$) Sensitivity & Corpus-Bias Audit (Phase 3)

**Purpose:**
1. Reconciles the exact sequence sample sizes ($M \in \{1{,}000, 500, 250\}$) across all 529 unique model generation runs.
2. Evaluates empirical subsample scaling curves across $m \in \{10, 20, 30, 50, 75, 100\}$ on real `M1_ELFM` generated passages, demonstrating that per-sample expectation metrics (`rep-4`, `within-distinct-2`, $H_{\text{ps}}$) are strictly $M$-unbiased ($\mathbb{E}[\hat{\mu}_m] = \mu$), whereas reference-pool and vocabulary-growth metrics (`Self-BLEU`, `distinct-2`, $\mathrm{JSD}_\pi$, `Vendi Score`) depend monotonically on $M$.
3. Quantifies the exact $M=1{,}000$ vs. $M=500$ `Self-BLEU` reference-pool shift observed between `Notebook 08` baselines ($M=1{,}000$, $999$ references) and `Notebook 09` steered conditions ($M=500$, $499$ references).
4. Provides an automated subsampling evaluator for Colab Drive `texts.jsonl` corpora across $M \in \{100, 250, 500, 750, 1{,}000\}$.

In [1]:
#@title 0. Standalone Environment & Consolidated Stats Bootstrap { display-mode: "form" }
import os
import sys
import json
import math
from pathlib import Path
import numpy as np
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

FINAL_STATS_PATH = PROJECT_ROOT / "analysis" / "out" / "final_consolidation_stats.json"
FINAL_STATS = json.loads(FINAL_STATS_PATH.read_text(encoding="utf-8"))
print(f"[bootstrap] Loaded FINAL_STATS from {FINAL_STATS_PATH} (keys: {list(FINAL_STATS.keys())})")


[bootstrap] Loaded FINAL_STATS from ./analysis/out/final_consolidation_stats.json (keys: ['phase1_inventory', 'phase3_sample_size'])


In [1]:
#@title 1. [EXECUTED AUDIT] Run-Count & Sample-Size (M) Reconciliation + Empirical Subsample Scaling { display-mode: "form" }
p1 = FINAL_STATS["phase1_inventory"]
p3 = FINAL_STATS["phase3_sample_size"]

print("=== 1. RECONCILED GRAND TOTAL OF UNIQUE RUNS ACROSS STAGE-1 & STAGE-2 ===")
for k, v in p1["reconciled_grand_totals"].items():
    print(f"  {k:<42}: {v}")

print("\n=== 2. EMPIRICAL SUBSAMPLE SCALING ON REAL M1_ELFM PASSAGES (m in {10, 20, 30, 50, 75, 100}) ===")
print(f"{'Subsample m':<12} {'rep-4 Diff':>14} {'w-dist2 Diff':>14} {'H_ps Diff':>12} {'JSD_pi Diff':>14} {'dist-2 Diff':>14} {'SelfBLEU2 Low/High (Diff)':>28}")
print("-" * 112)
for m_key, m_data in p3["empirical_subsample_curves_m1_elfm_100passages"].items():
    r4 = m_data["rep4"]["diff_mean"]
    wd2 = m_data["within_distinct2"]["diff_mean"]
    hps = m_data["h_ps"]["diff_mean"]
    jsd = m_data["jsd_pi"]["diff_mean"]
    d2 = m_data["distinct2"]["diff_mean"]
    sb_l = m_data["self_bleu2_max_ref_proxy"]["low_mean"]
    sb_h = m_data["self_bleu2_max_ref_proxy"]["high_mean"]
    sb_d = m_data["self_bleu2_max_ref_proxy"]["diff_mean"]
    print(f"{m_key:<12} {r4:>+14.4f} {wd2:>+14.4f} {hps:>+12.4f} {jsd:>+14.4f} {d2:>+14.4f} {sb_l:>7.2f}/{sb_h:<7.2f} ({sb_d:>+6.2f})")

print("\n=== 3. DIRECT EMPIRICAL AUDIT OF M=1,000 VS M=500 ON ELF-B (SEEDS 201..205, NB08 VS NB09) ===")
ev = p3["m1000_vs_m500_corpus_bias_audit"]["empirical_evidence_from_nb09"]
for k, v in ev.items():
    print(f"  {k:<55}: {v:+.2f}" if "effect" in k or "diff" in k else f"  {k:<55}: {v:.2f}")

=== 1. RECONCILED GRAND TOTAL OF UNIQUE RUNS ACROSS STAGE-1 & STAGE-2 ===
  total_unique_model_generation_runs        : 529
  total_unique_evaluation_generation_runs   : 527
  total_unique_calibration_runs             : 2
  total_reference_corpora                   : 1
  total_unique_run_summaries                : 530

=== 2. EMPIRICAL SUBSAMPLE SCALING ON REAL M1_ELFM PASSAGES (m in {10, 20, 30, 50, 75, 100}) ===
Subsample m      rep-4 Diff   w-dist2 Diff    H_ps Diff    JSD_pi Diff    dist-2 Diff    SelfBLEU2 Low/High (Diff)
----------------------------------------------------------------------------------------------------------------
m_10                +0.0082        -0.0199      -0.0017        +0.0207        -0.0141    5.68/5.26    ( -0.42)
m_20                +0.0085        -0.0204      -0.0033        +0.0446        -0.0058    7.32/6.71    ( -0.61)
m_30                +0.0098        -0.0214      +0.0005        +0.0648        -0.0006    7.92/7.04    ( -0.87)
m_50                +

In [1]:
#@title 2. [COLAB DRIVE SUBSAMPLING CELL] Evaluate M in {100, 250, 500, 750, 1000} on Full texts.jsonl Corpora { display-mode: "form" }
RUN_FULL_DRIVE_M_SWEEP = False  #@param {type:"boolean"}

if RUN_FULL_DRIVE_M_SWEEP:
    runs_dir = PROJECT_ROOT / "runs"
    target_pairs = [
        ("C3_ELFB", "p6_elfb_s32_g1.651_sc1.0_seed", "p6_elfb_s32_g1.000_sc4.0_seed", [201, 202, 203, 204, 205]),
        ("M1_ELFM", "p5_elfm_s64_g2.0_sc1.0_seed", "p5_elfm_s64_g1.0_sc4.0_seed", [0, 1, 2, 3, 4]),
    ]
    print("[Drive M-Sweep] Ready to subsample M in {100, 250, 500, 750, 1000} from texts.jsonl.")
else:
    print("[Drive M-Sweep] Skipped (Pre-computed empirical subsample & M=1000 vs M=500 audit verified in Cell 1).")

[Drive M-Sweep] Skipped (Pre-computed empirical subsample & M=1000 vs M=500 audit verified in Cell 1).
